In [1]:
!pip install --quiet "numpy<2.0.0" paddlepaddle==2.6.2 paddleocr==2.8.1 opencv-python-headless pillow matplotlib pymupdf pydantic

In [2]:
import cv2
import os
import fitz  # PyMuPDF
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from paddleocr import PaddleOCR
from pydantic import BaseModel, Field
from typing import List, Tuple, Dict, Any

# Clear out verbose initialization warnings from the deep learning framework
import logging
logging.getLogger("ppocr").setLevel(logging.ERROR)

In [3]:
class OCRLineMatch(BaseModel):
    """Represents a single extracted line of text with its bounding geometry."""
    text: str = Field(..., description="The translated text string.")
    confidence: float = Field(..., description="Model confidence value between 0.0 and 1.0.")
    bounding_box: List[List[float]] = Field(..., description="4-point polygon coordinates: [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]")

class OCRExtractionPayload(BaseModel):
    """The standardized output payload for downstream VLM routing layers."""
    filename: str
    page_index: int
    extracted_lines: List[OCRLineMatch]
    full_text_block: str


class EnterpriseOCREngine:
    """
    A wrapper class for PaddleOCR that handles runtime initialization,
    thread-safe model execution, and outputs structured, schema-validated payloads.
    """
    def __init__(self, language: str = 'en', use_gpu: bool = False):
        # For PaddleOCR 2.x, use 'use_gpu' and 'use_angle_cls'
        self.ocr = PaddleOCR(
            use_angle_cls=True,  # Back to standard 2.x argument
            lang=language,
            use_gpu=use_gpu
            # show_log=False is still removed since it's deprecated
        )

    def extract_from_ndarray(self, img_matrix: np.ndarray, source_name: str, page_num: int = 0) -> OCRExtractionPayload:
        """Executes raw text localization extraction across an in-memory image matrix array."""
        if img_matrix is None:
            raise ValueError("Provided image matrix cannot be None")

        # Run inference
        raw_results = self.ocr.ocr(img_matrix, cls=True)

        processed_lines = []
        text_accumulator = []

        # Handle cases where an image has zero text elements
        if raw_results and raw_results[0]:
            for text_block in raw_results[0]:
                coords = text_block[0]    # List of 4 coordinate lists
                text_data = text_block[1][0] # String character output
                conf_score = text_block[1][1] # Probability float

                # Construct type-validated structured payload object
                line_obj = OCRLineMatch(
                    text=text_data,
                    confidence=float(conf_score),
                    bounding_box=coords
                )
                processed_lines.append(line_obj)
                text_accumulator.append(text_data)

        return OCRExtractionPayload(
            filename=os.path.basename(source_name),
            page_index=page_num,
            extracted_lines=processed_lines,
            full_text_block="\n".join(text_accumulator)
        )

In [5]:
import cv2
import numpy as np
import fitz  # PyMuPDF
from PIL import Image, ImageDraw

# 1. Generate "apex_estimate.png"
estimate_img = np.ones((400, 700, 3), dtype=np.uint8) * 255
pil_img = Image.fromarray(estimate_img)
draw = ImageDraw.Draw(pil_img)

draw.text((30, 40), "APEX COLLISION REPAIR CENTER", fill=(0, 0, 0))
draw.text((30, 90), "PART NUMBER: 88921-X-FRONT BUMPER", fill=(50, 50, 50))
draw.text((30, 130), "LABOR CHARGES (HOURLY RATE) ... $450.00", fill=(50, 50, 50))
draw.text((30, 170), "TOTAL ESTIMATE VALUE ........ $1,250.00", fill=(10, 10, 10))

estimate_matrix = np.array(pil_img)
cv2.imwrite("apex_estimate.png", estimate_matrix)

# 2. Generate "pharmacy_batch.pdf"
pdf_doc = fitz.open()
p1 = pdf_doc.new_page(width=600, height=300)
p1.insert_text((40, 50), "PHARMACY INVOICE BATCH", fontsize=16)
p1.insert_text((40, 100), "PRESCRIPTION RX-55219: $85.00")

p2 = pdf_doc.new_page(width=600, height=300)
p2.insert_text((40, 50), "PHARMACY INVOICE BATCH", fontsize=16)
p2.insert_text((40, 100), "PRESCRIPTION RX-99021: $120.00")

pdf_doc.save("pharmacy_batch.pdf")
pdf_doc.close()

print("✅ Successfully generated 'apex_estimate.png' and 'pharmacy_batch.pdf' in your workspace!")

✅ Successfully generated 'apex_estimate.png' and 'pharmacy_batch.pdf' in your workspace!


In [6]:
# Instantiate our pipeline manager component
engine = EnterpriseOCREngine(use_gpu=False)

# Run extraction across our invoice image target
image_matrix = cv2.imread("apex_estimate.png")
image_payload = engine.extract_from_ndarray(image_matrix, source_name="apex_estimate.png")

print(f"File Source Analyzed: {image_payload.filename}")
print(f"Aggregated Text Block:\n{image_payload.full_text_block}\n")
print(f"Confidence Array Verification:")
for line in image_payload.extracted_lines:
    print(f" -> [{line.confidence:.4f}] : {line.text}")

File Source Analyzed: apex_estimate.png
Aggregated Text Block:
APEX COLLISION REPAIR CENTER
PART NUMBER 88921-X-FRONT BUMPER
LABORCHARGES (HOURLY RATE) ... $450.00
TOTAL ESTIMATE VALUE.....$1,250.00

Confidence Array Verification:
 -> [0.9664] : APEX COLLISION REPAIR CENTER
 -> [0.9813] : PART NUMBER 88921-X-FRONT BUMPER
 -> [0.9877] : LABORCHARGES (HOURLY RATE) ... $450.00
 -> [0.9400] : TOTAL ESTIMATE VALUE.....$1,250.00


In [7]:
pdf_path = "pharmacy_batch.pdf"
doc = fitz.open(pdf_path)

print(f"Beginning deep processing runs for: {pdf_path}")
for page_idx in range(len(doc)):
    page = doc[page_idx]
    # Rasterize current page to pixel buffer at 150 DPI matrix
    pix = page.get_pixmap(matrix=fitz.Matrix(150/72, 150/72))
    img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)

    # Standardize image array back to BGR for general OpenCV compliance
    if pix.n == 4:
        img_data = cv2.cvtColor(img_data, cv2.COLOR_RGBA2BGR)
    elif pix.n == 3:
        img_data = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)

    page_payload = engine.extract_from_ndarray(img_data, source_name=pdf_path, page_num=page_idx)

    print(f"\n--- Output Payload Page Index: {page_payload.page_index} ---")
    print(page_payload.full_text_block)

doc.close()

Beginning deep processing runs for: pharmacy_batch.pdf

--- Output Payload Page Index: 0 ---
PHARMACY INVOICE BATCHE
PRESCRIPTION RX-55219:$85.00

--- Output Payload Page Index: 1 ---
PHARMACY INVOICE BATCH
PRESCRIPTION RX-99021: $120.00


In [8]:
# Create a base clear image
clean_base = np.ones((150, 600, 3), dtype=np.uint8) * 255
pil_c = Image.fromarray(clean_base)
draw_c = ImageDraw.Draw(pil_c)
draw_c.text((20, 50), "ACCOUNT TOTAL BALANCE: $5,820.11", fill=(0,0,0))
clean_matrix = np.array(pil_c)

# Create an aggressively blurred version using a Gaussian Kernel
blurry_matrix = cv2.GaussianBlur(clean_matrix, (15, 15), 0)

# Extract text from both versions to compare accuracy
payload_clean = engine.extract_from_ndarray(clean_matrix, "clean.png")
payload_blurry = engine.extract_from_ndarray(blurry_matrix, "blurry.png")

print("--- OCR Performance Comparison ---")
print(f"Clean Target Extracted Text:   {payload_clean.full_text_block.strip()}")
print(f"Clean Extraction Confidence:    {np.mean([l.confidence for l in payload_clean.extracted_lines]):.4f}" if payload_clean.extracted_lines else "0.0")
print("---------------------------------")
print(f"Blurry Target Extracted Text:  {payload_blurry.full_text_block.strip()}")
print(f"Blurry Extraction Confidence:   {np.mean([l.confidence for l in payload_blurry.extracted_lines]):.4f}" if payload_blurry.extracted_lines else "0.0")

--- OCR Performance Comparison ---
Clean Target Extracted Text:   ACCOUNT TOTAL BALANCE: $5,820.11
Clean Extraction Confidence:    0.9932
---------------------------------
Blurry Target Extracted Text:  CO6R7 TO5 8/MEE: 8683511
Blurry Extraction Confidence:   0.5322
